# Lab 04: Spark SQL & Optimization

## Objectives
- Use Spark SQL for data analysis and manipulation
- Understand the Catalyst optimizer and query execution process
- Write optimized SQL queries using best practices
- Use query hints for manual optimization when needed
- Implement Adaptive Query Execution (AQE) for runtime optimization
- Analyze query plans and identify performance bottlenecks
- Apply performance tuning strategies for SQL workloads

## Domain Coverage
- **Spark SQL — 20%**
- **DataFrame API — 30%**

## Estimates
- **Time:** ~90-120 minutes
- **Difficulty:** Intermediate
- **Prerequisites:** Lab 01 (Spark Fundamentals), Lab 02 (Data Loading & Transformation)

![Spark SQL Optimization Diagram](../assets/diagrams/lab-04-spark-sql-optimization.mmd)

In [ ]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, sum, max, min, when, lit
from pyspark.sql.functions import broadcast
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

In [ ]:
# Create Spark session with optimization configurations
spark = SparkSession.builder \
    .appName("SparkSQLOptimization") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .getOrCreate()

# Verify Spark session
print(f"Spark version: {spark.version}")
print(f"AQE enabled: {spark.conf.get('spark.sql.adaptive.enabled')}")

## A. Creating Sample Data for SQL Operations

Let's create comprehensive sample data to practice SQL operations.

In [ ]:
# Create employees data
employees_data = [
    (1, "Alice Johnson", 28, 75000, 101, "Engineering"),
    (2, "Bob Smith", 35, 85000, 102, "Marketing"),
    (3, "Charlie Brown", 42, 95000, 101, "Engineering"),
    (4, "Diana Prince", 31, 80000, 103, "Sales"),
    (5, "Eve Williams", 29, 72000, 102, "Marketing"),
    (6, "Frank Davis", 38, 88000, 101, "Engineering"),
    (7, "Grace Miller", 33, 78000, 103, "Sales"),
    (8, "Henry Wilson", 45, 105000, 104, "Executive"),
    (9, "Ivy Chen", 27, 68000, 102, "Marketing"),
    (10, "Jack Taylor", 40, 92000, 101, "Engineering")
]

df_employees = spark.createDataFrame(employees_data, 
    ["id", "name", "age", "salary", "dept_id", "department"])
df_employees.createOrReplaceTempView("employees")

# Create departments data
departments_data = [
    (101, "Engineering", "Building A", 500000),
    (102, "Marketing", "Building B", 300000),
    (103, "Sales", "Building C", 400000),
    (104, "Executive", "Building D", 200000)
]

df_departments = spark.createDataFrame(departments_data,
    ["dept_id", "dept_name", "location", "budget"])
df_departments.createOrReplaceTempView("departments")

# Create sales data
sales_data = [
    ("2024-01-01", "Product A", "Electronics", 5, 500.00),
    ("2024-01-01", "Product B", "Electronics", 3, 300.00),
    ("2024-01-02", "Product C", "Furniture", 2, 400.00),
    ("2024-01-02", "Product D", "Furniture", 4, 800.00),
    ("2024-01-03", "Product A", "Electronics", 6, 600.00),
    ("2024-01-03", "Product E", "Office", 10, 200.00),
    ("2024-01-04", "Product B", "Electronics", 2, 200.00),
    ("2024-01-04", "Product C", "Furniture", 3, 600.00)
]

df_sales = spark.createDataFrame(sales_data,
    ["date", "product", "category", "quantity", "revenue"])
df_sales.createOrReplaceTempView("sales")

print("Sample data created and registered as temp views")
df_employees.show(5)
df_departments.show()
df_sales.show(5)

## B. Basic SQL Operations

Let's practice fundamental SQL operations.

In [ ]:
# SELECT with filtering
print("Employees with salary > 80000:")
spark.sql("""
    SELECT name, salary 
    FROM employees 
    WHERE salary > 80000
    ORDER BY salary DESC
""").show()

# SELECT with multiple conditions
print("\nEngineering employees aged 30-40:")
spark.sql("""
    SELECT name, age, department
    FROM employees
    WHERE department = 'Engineering' 
      AND age BETWEEN 30 AND 40
""").show()

In [ ]:
# Aggregations
print("Average salary by department:")
spark.sql("""
    SELECT department, 
           COUNT(*) as employee_count,
           AVG(salary) as avg_salary,
           SUM(salary) as total_salary,
           MAX(salary) as max_salary,
           MIN(salary) as min_salary
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC
""").show()

# HAVING clause
print("\nDepartments with average salary > 75000:")
spark.sql("""
    SELECT department, AVG(salary) as avg_salary
    FROM employees
    GROUP BY department
    HAVING AVG(salary) > 75000
""").show()

## C. Advanced SQL Features

Practice advanced SQL features like CTEs, window functions, and case statements.

In [ ]:
# Common Table Expression (CTE)
print("Using CTE for department statistics:")
spark.sql("""
    WITH dept_stats AS (
        SELECT department, 
               AVG(salary) as avg_salary,
               COUNT(*) as emp_count
        FROM employees
        GROUP BY department
    )
    SELECT e.name, e.salary, d.avg_salary, d.emp_count
    FROM employees e
    JOIN dept_stats d ON e.department = d.department
    WHERE e.salary > d.avg_salary
    ORDER BY e.salary DESC
""").show()

In [ ]:
# CASE statement
print("Salary categorization:")
spark.sql("""
    SELECT name, salary,
           CASE 
               WHEN salary < 70000 THEN 'Junior'
               WHEN salary < 90000 THEN 'Mid-Level'
               ELSE 'Senior'
           END as salary_level
    FROM employees
    ORDER BY salary DESC
""").show()

In [ ]:
# Window functions in SQL
print("Employee ranking within department:")
spark.sql("""
    SELECT name, department, salary,
           ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) as row_num,
           RANK() OVER (PARTITION BY department ORDER BY salary DESC) as rank_num,
           DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) as dense_rank
    FROM employees
    ORDER BY department, salary DESC
""").show()

## D. Query Execution Plans

Understanding query plans is crucial for optimization.

In [ ]:
# Simple query plan
print("Simple query plan:")
spark.sql("SELECT * FROM employees WHERE salary > 80000").explain()

# Complex query plan
print("\nComplex query plan with join:")
complex_query = spark.sql("""
    SELECT e.name, e.salary, d.dept_name, d.location
    FROM employees e
    JOIN departments d ON e.dept_id = d.dept_id
    WHERE e.salary > 75000
    ORDER BY e.salary DESC
""")
complex_query.explain()

In [ ]:
# Extended explain (logical + physical)
print("Extended query plan:")
complex_query.explain(extended=True)

## E. Query Hints

Query hints allow manual optimization when the optimizer doesn't choose the best plan.

In [ ]:
# Broadcast join hint
print("Broadcast join hint:")
broadcast_query = spark.sql("""
    SELECT /*+ BROADCAST(d) */ e.name, e.salary, d.dept_name
    FROM employees e
    JOIN departments d ON e.dept_id = d.dept_id
""")
broadcast_query.explain()
broadcast_query.show()

In [ ]:
# Repartition hint
print("Repartition hint:")
repartition_query = spark.sql("""
    SELECT /*+ REPARTITION(10) */ * FROM employees
""")
repartition_query.explain()

## F. Performance Optimization Techniques

Apply various optimization techniques to improve query performance.

In [ ]:
# Predicate pushdown (filter early)
print("Predicate pushdown optimization:")
# Bad: Filter after join
bad_query = spark.sql("""
    SELECT e.name, e.salary
    FROM employees e
    JOIN departments d ON e.dept_id = d.dept_id
    WHERE e.salary > 80000
""")

# Good: Filter before join
good_query = spark.sql("""
    SELECT e.name, e.salary
    FROM (SELECT * FROM employees WHERE salary > 80000) e
    JOIN departments d ON e.dept_id = d.dept_id
""")

print("Bad query plan:")
bad_query.explain()
print("\nGood query plan:")
good_query.explain()

In [ ]:
# Column pruning (select only needed columns)
print("Column pruning optimization:")
# Bad: SELECT *
bad_query = spark.sql("SELECT * FROM employees")

# Good: SELECT specific columns
good_query = spark.sql("SELECT name, salary FROM employees")

print("Bad query plan:")
bad_query.explain()
print("\nGood query plan:")
good_query.explain()

In [ ]:
# Caching frequently used DataFrames
print("Caching optimization:")
# Cache the employees DataFrame
spark.sql("CACHE TABLE employees")

# Check cached tables
print("\nCached tables:")
spark.sql("SHOW TABLES").show()

# Run query on cached table
spark.sql("SELECT COUNT(*) FROM employees").show()

# Uncache when done
spark.sql("UNCACHE TABLE employees")
print("\nTable uncached")

## G. Adaptive Query Execution (AQE)

AQE optimizes queries at runtime based on actual data statistics.

In [ ]:
# Check AQE configuration
print("AQE Configuration:")
print(f"AQE enabled: {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"Coalesce partitions: {spark.conf.get('spark.sql.adaptive.coalescePartitions.enabled')}")
print(f"Skew join optimization: {spark.conf.get('spark.sql.adaptive.skewJoin.enabled')}")

In [ ]:
# Demonstrate AQE benefits with skewed data
# Create skewed data
skewed_data = [(i, f"product_{i}", i % 5, 100.0 * (i % 10 + 1)) for i in range(1000)]
df_skewed = spark.createDataFrame(skewed_data, ["id", "name", "category", "price"])
df_skewed.createOrReplaceTempView("skewed_products")

# Query that benefits from AQE
aqe_query = spark.sql("""
    SELECT category, AVG(price) as avg_price
    FROM skewed_products
    GROUP BY category
""")

print("AQE optimized query:")
aqe_query.explain()
aqe_query.show()

## H. Catalog Operations

Manage databases, tables, and views using SQL.

In [ ]:
# List databases
print("Databases:")
spark.sql("SHOW DATABASES").show()

# Current database
print("\nCurrent database:")
spark.sql("SELECT current_database()").show()

# List tables
print("\nTables in current database:")
spark.sql("SHOW TABLES").show()

In [ ]:
# Describe table
print("Table structure:")
spark.sql("DESCRIBE employees").show()

# Drop temp view
spark.sql("DROP VIEW IF EXISTS skewed_products")
print("\nTemp view dropped")

## Exam-Style Review

**Q1.** What is the purpose of the Catalyst optimizer in Spark SQL?
- A) To manage cluster resources
- B) To optimize query execution plans through rule-based and cost-based optimization
- C) To handle data serialization
- D) To manage memory allocation

**Q2.** What is Adaptive Query Execution (AQE)?
- A) A query language for Spark
- B) Runtime optimization that adapts based on actual data statistics
- C) A type of join operation
- D) A data format

**Q3.** When should you use a broadcast join hint?
- A) When joining two large tables
- B) When one table is small enough to fit in memory
- C) When you need to sort the results
- D) When using window functions

**Q4.** What is predicate pushdown?
- A) Moving filters closer to the data source to reduce data volume
- B) Pushing data to multiple nodes
- C) A type of join operation
- D) A data compression technique

### Answers
- **Q1: B** — The Catalyst optimizer optimizes query execution plans through rule-based and cost-based optimization.
- **Q2: B** — AQE is runtime optimization that adapts query plans based on actual data statistics during execution.
- **Q3: B** — Use broadcast join hints when one table is small enough to fit in memory to avoid shuffling.
- **Q4: A** — Predicate pushdown moves filters closer to the data source to reduce the amount of data processed.

## Key Takeaways
- Spark SQL provides a familiar SQL interface for DataFrame operations
- The Catalyst optimizer automatically optimizes queries through multiple phases
- Query hints allow manual optimization when needed
- AQE provides runtime optimization based on actual data statistics
- Understanding query plans is essential for performance tuning
- Predicate pushdown and column pruning are key optimization techniques
- Caching frequently used DataFrames improves performance
- Catalog operations manage databases, tables, and views

**Next:** [Lab 05 — Spark Streaming Fundamentals](chapter-05-spark-streaming-fundamentals.ipynb)

In [ ]:
# Cleanup
spark.catalog.clearCache()
spark.sql("DROP VIEW IF EXISTS employees")
spark.sql("DROP VIEW IF EXISTS departments")
spark.sql("DROP VIEW IF EXISTS sales")
print("Lab 04 complete. Cache cleared and views dropped.")